"""

This is just the original md_jaccard_analysis.ipynb but now sweeps through all the params from the original config 
for all three threshold options (distance, distance_change, knn)

"""

In [15]:
from dataclasses import replace
from pathlib import Path
import numpy as np
import torch
from itertools import product

import os, sys
sys.path.append("/home/akapociu/ift/interactiondynamics")
print(os.getcwd())
print(sys.path[-1])

out_path = Path("/home/akapociu/ift/interactiondynamics/results/md22_full_calibration.jsonl")
if out_path.exists():
    out_path.unlink()

/home/akapociu/ift/interactiondynamics/analysis
/home/akapociu/ift/interactiondynamics


In [16]:
from data.md22_binned import MD22BinnedConfig, MD22BinnedDataset

In [17]:
def append_jsonl(path, row):
    import json
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

def edge_set(batch):
    return set(zip(batch.src.cpu().tolist(), batch.dst.cpu().tolist()))


def mean_jaccard_for_dataset(ds, split="train"):
    bins = list(ds.bins(split))
    edge_sets = [edge_set(batch) for batch in bins]

    if len(edge_sets) < 2:
        return None

    vals = []
    for prev, curr in zip(edge_sets[:-1], edge_sets[1:]):
        inter = len(prev & curr)
        union = len(prev | curr)
        vals.append(inter / union if union > 0 else 1.0)

    return float(np.mean(vals))


def avg_edges_per_bin(ds, split="train"):
    bins = list(ds.bins(split))
    if len(bins) == 0:
        return None
    return float(np.mean([batch.src.numel() for batch in bins]))


# returns list of dictionaries
# only creates VALID combos for each threshold mode yay
def build_md22_param_grid(
    *,
    event_modes=("distance", "distance_change", "knn"),
    distance_thresholds=(2.0, 3.0, 5.0),
    distance_change_thresholds=(0.01, 0.05, 0.1),
    distance_change_use_absolute_options=(True,),
    knn_ks=(2, 4, 8),
    observation_noise_pos_options=(0.0, 0.01, 0.02),
):
    combos = []

    for event_mode in event_modes:
        if event_mode == "distance":
            for dist_thr, noise_pos in product(
                distance_thresholds,
                observation_noise_pos_options,
            ):
                combos.append({
                    "event_mode": "distance",
                    "distance_threshold": float(dist_thr),
                    "observation_noise_pos": float(noise_pos),
                })

        elif event_mode == "distance_change":
            for dchg_thr, use_abs, noise_pos in product(
                distance_change_thresholds,
                distance_change_use_absolute_options,
                observation_noise_pos_options,
            ):
                combos.append({
                    "event_mode": "distance_change",
                    "distance_change_threshold": float(dchg_thr),
                    "distance_change_use_absolute": bool(use_abs),
                    "observation_noise_pos": float(noise_pos),
                })

        elif event_mode == "knn":
            for k, noise_pos in product(
                knn_ks,
                observation_noise_pos_options,
            ):
                combos.append({
                    "event_mode": "knn",
                    "knn_k": int(k),
                    "observation_noise_pos": float(noise_pos),
                })

        else:
            raise ValueError(f"Unknown event_mode: {event_mode}")

    return combos

# MAIN SWEEP FUNCTION
# every param combo from the grid ^^ + every stride from candidate_strides
def calibrate_md22_config(
    npz_path,
    *,
    device,
    base_cfg=None,
    candidate_strides=(32, 64, 128, 256, 512, 1024, 2056),
    param_grid=None,
    target_low=0.55,
    target_high=0.80,
    min_total_bins=256,
    split="train",
):
    """
    Sweep over BOTH:
      - frame_stride
      - MD22 event extraction / noise hyperparameters

    No max_total_bins cap: each stride uses the full trajectory.
    Returns:
      best_cfg, diagnostics_rows
    """
    stem = Path(npz_path).stem

    # base config if you don't specify but of course 
    if base_cfg is None:
        base_cfg = MD22BinnedConfig(
            name=stem,
            npz_path=str(npz_path),
            event_mode="distance",
            distance_threshold=5.0,
            distance_change_threshold=0.1,
            distance_change_use_absolute=True,
            knn_k=4,
            observation_noise_pos=0.0,
            observation_noise_force=0.0,
            min_edges_per_bin=1,
            device=device,
        )

    # same ^ but for param grid
    if param_grid is None:
        param_grid = build_md22_param_grid()

    rows = []

    # every param/stride combo
    for params in param_grid:
        for stride in candidate_strides:
            cfg = replace( 
                base_cfg,
                name=stem,
                npz_path=str(npz_path),
                frame_stride=int(stride),
                **params, # uses all param options from grid
            )

            try:
                ds = MD22BinnedDataset(cfg) # build it
                spec = ds.spec()
                mj = mean_jaccard_for_dataset(ds, split=split)
                avg_edges = avg_edges_per_bin(ds, split=split)

                row = {
                    "molecule": stem,
                    "stride": int(stride),
                    "event_mode": cfg.event_mode,
                    "distance_threshold": getattr(cfg, "distance_threshold", None),
                    "distance_change_threshold": getattr(cfg, "distance_change_threshold", None),
                    "distance_change_use_absolute": getattr(cfg, "distance_change_use_absolute", None),
                    "knn_k": getattr(cfg, "knn_k", None),
                    "observation_noise_pos": cfg.observation_noise_pos,
                    "observation_noise_force": cfg.observation_noise_force,
                    "num_bins": int(spec.num_bins),
                    "num_nodes": int(spec.num_nodes),
                    "mean_jaccard": mj,
                    "avg_edges_per_bin": avg_edges,
                    "valid_bins": (spec.num_bins is not None and spec.num_bins >= min_total_bins),
                }
                rows.append(row)

            except Exception as e:
                rows.append({
                    "molecule": stem,
                    "stride": int(stride),
                    "event_mode": params.get("event_mode"),
                    "distance_threshold": params.get("distance_threshold"),
                    "distance_change_threshold": params.get("distance_change_threshold"),
                    "distance_change_use_absolute": params.get("distance_change_use_absolute"),
                    "knn_k": params.get("knn_k"),
                    "observation_noise_pos": params.get("observation_noise_pos"),
                    "num_bins": None,
                    "num_nodes": None,
                    "mean_jaccard": None,
                    "avg_edges_per_bin": None,
                    "valid_bins": False,
                    "error": f"{type(e).__name__}: {e}",
                })

    valid = [
        r for r in rows
        if r["mean_jaccard"] is not None and r["valid_bins"]
    ]

    if not valid:
        raise RuntimeError(f"No valid config candidates for {stem}")

    in_band = [
        r for r in valid
        if target_low <= r["mean_jaccard"] <= target_high
    ]

    target_mid = 0.5 * (target_low + target_high)

    def score(r):
        return (
            abs(r["mean_jaccard"] - target_mid),  # primary
            r["stride"],                          # prefer smaller stride if tied
            -r["num_bins"],                       # prefer more bins
        )

    chosen = sorted(in_band if in_band else valid, key=score)[0]

    best_cfg = replace(
        base_cfg,
        name=stem,
        npz_path=str(npz_path),
        frame_stride=int(chosen["stride"]),
        event_mode=chosen["event_mode"],
        distance_threshold=chosen["distance_threshold"] if chosen["distance_threshold"] is not None else base_cfg.distance_threshold,
        distance_change_threshold=chosen["distance_change_threshold"] if chosen["distance_change_threshold"] is not None else base_cfg.distance_change_threshold,
        distance_change_use_absolute=chosen["distance_change_use_absolute"] if chosen["distance_change_use_absolute"] is not None else base_cfg.distance_change_use_absolute,
        knn_k=chosen["knn_k"] if chosen["knn_k"] is not None else base_cfg.knn_k,
        observation_noise_pos=chosen["observation_noise_pos"],
        observation_noise_force=chosen["observation_noise_force"],
    )

    return best_cfg, rows

In [18]:
md22_npz_paths = [
    "/home/akapociu/ift/interactiondynamics/data/MD_DATA/uracil.npz",
    "/home/akapociu/ift/interactiondynamics/data/MD_DATA/naphthalene.npz",
    "/home/akapociu/ift/interactiondynamics/data/MD_DATA/stachyose.npz",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

chosen_cfgs = []

param_grid = build_md22_param_grid(
    event_modes=("distance", "distance_change", "knn"),
    distance_thresholds=(2.0, 3.0, 5.0),
    distance_change_thresholds=(0.05, 0.1, 0.2),
    distance_change_use_absolute_options=(True, ),
    knn_ks=(8, 12, 32, 64),
    observation_noise_pos_options=(0.0, 0.01, 0.02),
)

for npz_path in md22_npz_paths:
    best_cfg, rows = calibrate_md22_config(
        npz_path,
        device=device,
        base_cfg=MD22BinnedConfig(
            name=Path(npz_path).stem,
            npz_path=npz_path,
            event_mode="distance",
            distance_threshold=5.0,
            distance_change_threshold=0.1,
            distance_change_use_absolute=True,
            knn_k=4,
            observation_noise_pos=0.0,
            observation_noise_force=0.0,
            min_edges_per_bin=1,
            device=device,
        ),
        candidate_strides=(32, 64, 128, 256, 512, 1024, 2056),
        param_grid=param_grid,
        target_low=0.55,
        target_high=0.80,
        min_total_bins=256,
    )

    chosen_cfgs.append(best_cfg)

    for row in rows:
        append_jsonl(out_path, row),

    print(
        f"{Path(npz_path).stem}: "
        f"mode={best_cfg.event_mode}, "
        f"stride={best_cfg.frame_stride}, "
        f"dist_thr={best_cfg.distance_threshold}, "
        f"dchg_thr={best_cfg.distance_change_threshold}, "
        f"use_abs={best_cfg.distance_change_use_absolute}, "
        f"knn_k={best_cfg.knn_k}, "
        f"noise_pos={best_cfg.observation_noise_pos}, "
    )

KeyboardInterrupt: 